In [2]:
import pandas as pd
from pathlib import Path

# Project folders
raw_folder = Path("../data/raw")
cleaned_folder = Path("../data/cleaned")

# Create the cleaned folder if it doesn't exist
cleaned_folder.mkdir(parents=True, exist_ok=True)

# Find all yearly CSV files
raw_files = sorted(raw_folder.glob("amazon_india_*.csv"))

print("Number of raw datasets:", len(raw_files))

# Compare dataset shapes and column names
for file in raw_files:
    df = pd.read_csv(file)
    print(
        file.name,
        "| Shape:", df.shape,
        "| Columns:", len(df.columns)
    )

Number of raw datasets: 11
amazon_india_2015.csv | Shape: (33165, 34) | Columns: 34
amazon_india_2016.csv | Shape: (55275, 34) | Columns: 34
amazon_india_2017.csv | Shape: (77385, 34) | Columns: 34
amazon_india_2018.csv | Shape: (99495, 34) | Columns: 34
amazon_india_2019.csv | Shape: (121605, 34) | Columns: 34
amazon_india_2020.csv | Shape: (143715, 34) | Columns: 34
amazon_india_2021.csv | Shape: (138187, 34) | Columns: 34
amazon_india_2022.csv | Shape: (132660, 34) | Columns: 34
amazon_india_2023.csv | Shape: (127132, 34) | Columns: 34
amazon_india_2024.csv | Shape: (121605, 34) | Columns: 34
amazon_india_2025.csv | Shape: (77385, 34) | Columns: 34


In [3]:
for file in raw_files:
    df = pd.read_csv(file)

    print(f"\n--- {file.name} ---")
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print("Duplicate transaction IDs:",
          df["transaction_id"].duplicated().sum())
    print("Data types:")
    print(df.dtypes.value_counts())


--- amazon_india_2015.csv ---
Missing values: 39180
Duplicate rows: 0
Duplicate transaction IDs: 0
Data types:
str        23
float64     7
int64       4
Name: count, dtype: int64

--- amazon_india_2016.csv ---
Missing values: 65633
Duplicate rows: 0
Duplicate transaction IDs: 0
Data types:
str        23
float64     7
int64       4
Name: count, dtype: int64

--- amazon_india_2017.csv ---
Missing values: 91885
Duplicate rows: 0
Duplicate transaction IDs: 0
Data types:
str        23
float64     7
int64       4
Name: count, dtype: int64

--- amazon_india_2018.csv ---
Missing values: 118044
Duplicate rows: 0
Duplicate transaction IDs: 0
Data types:
str        23
float64     7
int64       4
Name: count, dtype: int64

--- amazon_india_2019.csv ---
Missing values: 143931
Duplicate rows: 0
Duplicate transaction IDs: 0
Data types:
str        23
float64     7
int64       4
Name: count, dtype: int64

--- amazon_india_2020.csv ---
Missing values: 172302
Duplicate rows: 0
Duplicate transaction IDs:

In [4]:
for file in raw_files:
    df = pd.read_csv(file)

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n--- {file.name} ---")
    print(missing.sort_values(ascending=False))


--- amazon_india_2015.csv ---
festival_name         22581
customer_rating        9969
customer_age_group     3976
delivery_charges       2654
dtype: int64

--- amazon_india_2016.csv ---
festival_name         37815
customer_rating       16764
customer_age_group     6630
delivery_charges       4424
dtype: int64

--- amazon_india_2017.csv ---
festival_name         52831
customer_rating       23592
customer_age_group     9273
delivery_charges       6189
dtype: int64

--- amazon_india_2018.csv ---
festival_name         67918
customer_rating       30234
customer_age_group    11939
delivery_charges       7953
dtype: int64

--- amazon_india_2019.csv ---
festival_name         82822
customer_rating       36793
customer_age_group    14595
delivery_charges       9721
dtype: int64

--- amazon_india_2020.csv ---
festival_name         99915
customer_rating       43625
customer_age_group    17244
delivery_charges      11518
dtype: int64

--- amazon_india_2021.csv ---
festival_name         95887
custo

In [5]:
def clean_missing_values(df):
    df = df.copy()

    # Fill missing age groups
    df["customer_age_group"] = (
        df["customer_age_group"].fillna("Unknown")
    )

    # Preserve missing customer ratings
    # Do not invent customer ratings

    # Fill missing festival names only for non-festival sales
    festival_flag = (
        df["is_festival_sale"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    non_festival = festival_flag.isin(["0", "false", "no"])

    df.loc[
        non_festival & df["festival_name"].isna(),
        "festival_name"
    ] = "Not Applicable"

    return df

In [6]:
def clean_festival_flag(df):
    df = df.copy()

    festival_mapping = {
        "1": True,
        "TRUE": True,
        "True": True,
        "Yes": True,
        "0": False,
        "FALSE": False,
        "False": False,
        "No": False
    }

    df["is_festival_sale"] = (
        df["is_festival_sale"]
        .astype(str)
        .str.strip()
        .map(festival_mapping)
    )

    return df

In [7]:
df_2018 = pd.read_csv("../data/raw/amazon_india_2018.csv")

df_2018_clean = clean_festival_flag(df_2018)

print(
    df_2018_clean["is_festival_sale"]
    .value_counts(dropna=False)
)

is_festival_sale
False    67918
True     31577
Name: count, dtype: int64


In [8]:
df_2018_clean = clean_festival_flag(df_2018_clean)

df_2018_clean = clean_missing_values(df_2018_clean)

print("Missing age groups:",
      df_2018_clean["customer_age_group"].isna().sum())

print("Unknown age groups:",
      (df_2018_clean["customer_age_group"] == "Unknown").sum())

print("Missing festival names:",
      df_2018_clean["festival_name"].isna().sum())

print("Missing customer ratings:",
      df_2018_clean["customer_rating"].isna().sum())

Missing age groups: 0
Unknown age groups: 11939
Missing festival names: 0
Missing customer ratings: 30234


In [9]:
festival_name_issue = (
    df_2018_clean["is_festival_sale"]
    & df_2018_clean["festival_name"].eq("Not Applicable")
)

print("Festival sales marked Not Applicable:",
      festival_name_issue.sum())

Festival sales marked Not Applicable: 0


In [10]:
def clean_customer_rating(df):
    df = df.copy()

    df["customer_rating"] = (
        df["customer_rating"]
        .astype("string")
        .str.replace(" stars", "", regex=False)
        .str.split("/")
        .str[0]
    )

    df["customer_rating"] = pd.to_numeric(
        df["customer_rating"],
        errors="coerce"
    )

    return df

In [11]:

def clean_delivery_charges(df):
    df = df.copy()

    # Fill missing delivery charges with 0
    # only when final amount equals subtotal
    missing_charges = df["delivery_charges"].isna()

    charges_are_zero = (
        df.loc[missing_charges, "final_amount_inr"]
        == df.loc[missing_charges, "subtotal_inr"]
    )

    rows_to_fill = df.loc[missing_charges].index[charges_are_zero]

    df.loc[rows_to_fill, "delivery_charges"] = 0

    return df


In [12]:

df_2018_clean = clean_delivery_charges(df_2018_clean)

print("Missing delivery charges:",
      df_2018_clean["delivery_charges"].isna().sum())

print("Delivery charges equal to zero:",
      (df_2018_clean["delivery_charges"] == 0).sum())


Missing delivery charges: 0
Delivery charges equal to zero: 99495


In [13]:

print("Missing customer ratings:",
      df_2018_clean["customer_rating"].isna().sum())

print("Rating values:")
print(sorted(df_2018_clean["customer_rating"].dropna().unique()))


Missing customer ratings: 30234
Rating values:
['3', '3.0', '3.0 stars', '3.0/5.0', '3.5', '3.5 stars', '3.5/5.0', '3/5', '4', '4.0', '4.0 stars', '4.0/5.0', '4.5', '4.5 stars', '4.5/5.0', '4/5', '5', '5.0', '5.0 stars', '5.0/5.0', '5/5']


In [14]:
df_2018_clean = clean_customer_rating(df_2018_clean)

In [15]:
print("Missing customer ratings:",
      df_2018_clean["customer_rating"].isna().sum())

print("Rating values:")
print(sorted(df_2018_clean["customer_rating"].dropna().unique()))

Missing customer ratings: 30234
Rating values:
[np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]


In [16]:

def clean_original_price(df):
    df = df.copy()

    df["original_price_inr"] = (
        df["original_price_inr"]
        .astype("string")
        .str.replace("₹", "", regex=False)
        .str.replace("Rs.", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    df["original_price_inr"] = pd.to_numeric(
        df["original_price_inr"],
        errors="coerce"
    )

    return df


In [17]:
df_2018_clean = clean_original_price(df_2018_clean)

print(df_2018_clean["original_price_inr"].dtype)
print("Missing prices:",
      df_2018_clean["original_price_inr"].isna().sum())
print(df_2018_clean["original_price_inr"].describe())

Float64
Missing prices: 2982
count          96513.0
mean      87590.894402
std      421015.949115
min         -251527.09
25%            30661.0
50%           48733.08
75%          104592.57
max         24237111.0
Name: original_price_inr, dtype: Float64


In [18]:
negative_prices = df_2018_clean[
    df_2018_clean["original_price_inr"] < 0
]

print("Number of negative prices:", len(negative_prices))

print(negative_prices[
    [
        "product_name",
        "original_price_inr",
        "discounted_price_inr",
        "quantity",
        "subtotal_inr",
        "final_amount_inr"
    ]
].head(10))

Number of negative prices: 251
                         product_name  original_price_inr  \
1086  Motorola Moto G4 Plus 16GB Blue           -23099.28   
1841    Lenovo VivoBook 4GB RAM Black           -43348.81   
2304      ASUS Inspiron 4GB RAM Black          -138070.49   
2785    OnePlus OnePlus 5T 64GB Black           -89305.36   
2963        Xiaomi Redmi 5A 64GB Blue           -40339.02   
3809     OnePlus OnePlus 3T 64GB Blue          -102291.81   
3959          Xiaomi Mi A2 64GB Black           -47867.34   
5315         Apple iPhone 8 16GB Blue          -148508.92   
6394       Realme Realme 1 64GB White           -37554.97   
7513         HP Gaming 8GB RAM Silver          -114816.88   

      discounted_price_inr  quantity  subtotal_inr  final_amount_inr  
1086              23099.28         2      46198.56          46198.56  
1841              43348.81         2      86697.62          86697.62  
2304             138070.49         1     138070.49         138070.49  
2785         

In [19]:

negative_prices = df_2018_clean[
    df_2018_clean["original_price_inr"] < 0
]

matches_discounted = (
    negative_prices["original_price_inr"].abs()
    .round(2)
    == negative_prices["discounted_price_inr"].round(2)
)

print("Total negative prices:", len(negative_prices))
print("Absolute price matches discounted price:", matches_discounted.sum())
print("Does not match:", (~matches_discounted).sum())


Total negative prices: 251
Absolute price matches discounted price: 100
Does not match: 151


In [20]:

df_2018_clean.loc[
    df_2018_clean["original_price_inr"] < 0,
    "original_price_inr"
] = df_2018_clean.loc[
    df_2018_clean["original_price_inr"] < 0,
    "original_price_inr"
].abs()

print(
    "Remaining negative prices:",
    (df_2018_clean["original_price_inr"] < 0).sum()
)


Remaining negative prices: 0


In [21]:
scale_check = df_2018_clean[
    df_2018_clean["original_price_inr"].notna()
    & df_2018_clean["discounted_price_inr"].notna()
    & (df_2018_clean["discounted_price_inr"] != 0)
].copy()

scale_check["price_ratio"] = (
    scale_check["original_price_inr"]
    / scale_check["discounted_price_inr"]
)

print("Rows where original price is about 100 times discounted price:",
      scale_check["price_ratio"].between(99, 101).sum())

print(scale_check.nlargest(10, "price_ratio")[
    ["product_name", "original_price_inr",
     "discounted_price_inr", "price_ratio"]
])

Rows where original price is about 100 times discounted price: 100
                           product_name  original_price_inr  \
29591       Xiaomi Redmi 5A 256GB Black           4907797.0   
85135  Samsung Galaxy Note 9 128GB Blue          14224889.0   
61019           Oppo R17 Pro 64GB White           1752817.0   
30957         Apple iPhone 6 64GB Black          13903809.0   
7168     Apple iPhone 6 Plus 16GB Black          13427816.0   
83697    Xiaomi Redmi Note 3 64GB Black           2634322.0   
30049     OnePlus OnePlus 6 128GB White           7145654.0   
72398     OnePlus OnePlus 5T 32GB White          11283790.0   
18827     OnePlus OnePlus 6 128GB Black          10973981.0   
76917              Vivo Y95 128GB White           3897953.0   

       discounted_price_inr  price_ratio  
29591              14786.85   331.902806  
85135              48396.78    293.92222  
61019               6306.44   277.940803  
30957              52184.69   266.434638  
7168               51867

In [22]:
scale_check["original_div_100"] = (
    scale_check["original_price_inr"] / 100
)

scale_check["diff_after_div_100"] = (
    scale_check["original_div_100"]
    - scale_check["discounted_price_inr"]
).abs()

print(
    "Rows matching after dividing by 100:",
    (scale_check["diff_after_div_100"] < 1).sum()
)

Rows matching after dividing by 100: 100


In [23]:

# Identify rows where dividing original price by 100
# brings it within ₹1 of the discounted price
scale_error_mask = (
    df_2018_clean["original_price_inr"].notna()
    & df_2018_clean["discounted_price_inr"].notna()
    & (
        (
            df_2018_clean["original_price_inr"] / 100
            - df_2018_clean["discounted_price_inr"]
        ).abs() < 1
    )
)

print("Rows to correct:", scale_error_mask.sum())

# Correct the identified original prices
df_2018_clean.loc[
    scale_error_mask, "original_price_inr"
] = (
    df_2018_clean.loc[
        scale_error_mask, "original_price_inr"
    ] / 100
)

print("Remaining matching scale errors:",
      (
          (
              df_2018_clean["original_price_inr"] / 100
              - df_2018_clean["discounted_price_inr"]
          ).abs() < 1
      ).sum())


Rows to correct: 100
Remaining matching scale errors: 0


In [24]:
print(df_2018_clean["original_price_inr"].describe())

print("Negative prices:",
      (df_2018_clean["original_price_inr"] < 0).sum())

print("Missing original prices:",
      df_2018_clean["original_price_inr"].isna().sum())

count          96513.0
mean      81079.956609
std      324195.418504
min            2202.13
25%           31018.55
50%           48733.08
75%          104592.57
max         24237111.0
Name: original_price_inr, dtype: Float64
Negative prices: 0
Missing original prices: 2982


In [25]:
df_2018_clean.to_csv(
    "../data/cleaned/amazon_india_2018_clean.csv",
    index=False
)

print("2018 cleaned dataset saved successfully.")

2018 cleaned dataset saved successfully.


In [26]:

def clean_amazon_dataset(df):
    df = df.copy()

    # 1. Standardize festival flag
    df = clean_festival_flag(df)

    # 2. Clean missing values
    df = clean_missing_values(df)

    # 3. Clean customer ratings
    df = clean_customer_rating(df)

    # 4. Clean delivery charges
    df = clean_delivery_charges(df)

    # 5. Clean original prices
    df = clean_original_price(df)

    # 6. Correct negative original prices
    negative_mask = df["original_price_inr"] < 0
    df.loc[negative_mask, "original_price_inr"] = (
        df.loc[negative_mask, "original_price_inr"].abs()
    )

    return df


In [27]:

df_2018_raw = pd.read_csv("../data/raw/amazon_india_2018.csv")

df_2018_test = clean_amazon_dataset(df_2018_raw)

print("Raw shape:", df_2018_raw.shape)
print("Cleaned shape:", df_2018_test.shape)
print("Missing customer ratings:",
      df_2018_test["customer_rating"].isna().sum())
print("Negative original prices:",
      (df_2018_test["original_price_inr"] < 0).sum())


Raw shape: (99495, 34)
Cleaned shape: (99495, 34)
Missing customer ratings: 30234
Negative original prices: 0


In [28]:

def clean_amazon_dataset(df):
    df = df.copy()

    # 1. Standardize festival flag
    df = clean_festival_flag(df)

    # 2. Clean missing values
    df = clean_missing_values(df)

    # 3. Clean customer ratings
    df = clean_customer_rating(df)

    # 4. Clean delivery charges
    df = clean_delivery_charges(df)

    # 5. Clean original prices
    df = clean_original_price(df)

    # 6. Correct negative original prices
    negative_mask = df["original_price_inr"] < 0
    df.loc[negative_mask, "original_price_inr"] = (
        df.loc[negative_mask, "original_price_inr"].abs()
    )

    # 7. Correct potential scale errors:
    # Divide by 100 only when the adjusted original price
    # is within ₹1 of the discounted price
    scale_error_mask = (
        df["original_price_inr"].notna()
        & df["discounted_price_inr"].notna()
        & (
            (
                df["original_price_inr"] / 100
                - df["discounted_price_inr"]
            ).abs() < 1
        )
    )

    df.loc[scale_error_mask, "original_price_inr"] = (
        df.loc[scale_error_mask, "original_price_inr"] / 100
    )

    return df


In [29]:
df_2018_test = clean_amazon_dataset(df_2018_raw)

print("Rows:", len(df_2018_test))
print("Negative prices:",
      (df_2018_test["original_price_inr"] < 0).sum())
print("Missing original prices:",
      df_2018_test["original_price_inr"].isna().sum())

Rows: 99495
Negative prices: 0
Missing original prices: 2982


In [30]:

def standardize_categories(df):
    df = df.copy()

    category_mapping = {
        "ELECTRONICS": "Electronics",
        "Electronic": "Electronics",
        "Electronicss": "Electronics"
    }

    df["category"] = (
        df["category"]
        .astype("string")
        .str.strip()
        .replace(category_mapping)
    )

    return df


In [31]:

def standardize_cities(df):
    df = df.copy()

    city_mapping = {
        "Bangalore": "Bengaluru",
        "Banglore": "Bengaluru",
        "Bengalore": "Bengaluru",
        "BANGALORE": "Bengaluru",
        "chenai": "Chennai",
        "CHENNAI": "Chennai",
        "Madras": "Chennai",
        "delhi": "Delhi",
        "DELHI": "Delhi",
        "KOLKATA": "Kolkata",
        "kolkata": "Kolkata",
        "Calcutta": "Kolkata",
        "MUMBAI": "Mumbai",
        "mumba": "Mumbai"
    }

    df["customer_city"] = (
        df["customer_city"]
        .astype("string")
        .str.strip()
        .replace(city_mapping)
    )

    return df


In [32]:

def clean_amazon_dataset(df):
    df = df.copy()

    df = clean_festival_flag(df)
    df = clean_missing_values(df)
    df = clean_customer_rating(df)
    df = clean_delivery_charges(df)
    df = clean_original_price(df)

    negative_mask = df["original_price_inr"] < 0
    df.loc[negative_mask, "original_price_inr"] = (
        df.loc[negative_mask, "original_price_inr"].abs()
    )

    scale_error_mask = (
        df["original_price_inr"].notna()
        & df["discounted_price_inr"].notna()
        & (
            (
                df["original_price_inr"] / 100
                - df["discounted_price_inr"]
            ).abs() < 1
        )
    )

    df.loc[scale_error_mask, "original_price_inr"] = (
        df.loc[scale_error_mask, "original_price_inr"] / 100
    )

    df = standardize_categories(df)
    df = standardize_cities(df)

    return df


In [33]:

def clean_order_dates(df):
    df = df.copy()

    # Keep the original dates for investigation
    original_dates = df["order_date"].astype("string").str.strip()

    # Try parsing dates in the standard format first
    parsed_dates = pd.to_datetime(
        original_dates,
        errors="coerce",
        format="mixed",
        dayfirst=False
    )

    # Try month/day/year for dates that did not parse
    missing_dates = parsed_dates.isna()
    parsed_dates.loc[missing_dates] = pd.to_datetime(
        original_dates.loc[missing_dates],
        errors="coerce",
        format="%m/%d/%Y"
    )

    df["order_date"] = parsed_dates

    # Derive date fields from the cleaned date
    df["order_month"] = df["order_date"].dt.month
    df["order_year"] = df["order_date"].dt.year
    df["order_quarter"] = df["order_date"].dt.quarter

    return df


In [34]:
df = clean_order_dates(df)

In [36]:

df_2018_test = clean_amazon_dataset(df_2018_raw)

print("Rows:", len(df_2018_test))
print("Missing order dates:", df_2018_test["order_date"].isna().sum())
print("Month mismatches:", (
    df_2018_test["order_month"]
    != df_2018_test["order_date"].dt.month
).sum())
print("Year mismatches:", (
    df_2018_test["order_year"]
    != df_2018_test["order_date"].dt.year
).sum())
print("Quarter mismatches:", (
    df_2018_test["order_quarter"]
    != df_2018_test["order_date"].dt.quarter
).sum())


Rows: 99495
Missing order dates: 0


AttributeError: Can only use .dt accessor with datetimelike values

In [37]:
print(df_2018_test["order_date"].dtype)

str


In [38]:
df_2018_test["order_date"] = pd.to_datetime(
    df_2018_test["order_date"],
    errors="coerce"
)

print(df_2018_test["order_date"].dtype)

datetime64[us]


In [39]:
print("Missing order dates:",
      df_2018_test["order_date"].isna().sum())

print("Month mismatches:",
      (df_2018_test["order_month"]
       != df_2018_test["order_date"].dt.month).sum())

print("Year mismatches:",
      (df_2018_test["order_year"]
       != df_2018_test["order_date"].dt.year).sum())

print("Quarter mismatches:",
      (df_2018_test["order_quarter"]
       != df_2018_test["order_date"].dt.quarter).sum())

Missing order dates: 9574
Month mismatches: 9574
Year mismatches: 9574
Quarter mismatches: 9574


In [40]:
df_2018_raw = pd.read_csv("../data/raw/amazon_india_2018.csv")

In [41]:
df_2018_test = clean_amazon_dataset(df_2018_raw)

In [42]:
df = clean_order_dates(df)

In [43]:
print("Date data type:", df_2018_test["order_date"].dtype)

print("Missing order dates:",
      df_2018_test["order_date"].isna().sum())

Date data type: str
Missing order dates: 0


In [44]:
print("Month mismatches:",
      (df_2018_test["order_month"]
       != df_2018_test["order_date"].dt.month).sum())

print("Year mismatches:",
      (df_2018_test["order_year"]
       != df_2018_test["order_date"].dt.year).sum())

print("Quarter mismatches:",
      (df_2018_test["order_quarter"]
       != df_2018_test["order_date"].dt.quarter).sum())

AttributeError: Can only use .dt accessor with datetimelike values

In [45]:

def clean_order_dates(df):
    df = df.copy()

    original_dates = df["order_date"].astype("string").str.strip()

    # Parse the mixed date formats
    parsed_dates = pd.to_datetime(
        original_dates,
        errors="coerce",
        format="mixed",
        dayfirst=False
    )

    # Retry failed dates using month/day/year
    missing_dates = parsed_dates.isna()

    parsed_dates.loc[missing_dates] = pd.to_datetime(
        original_dates.loc[missing_dates],
        errors="coerce",
        format="%m/%d/%Y"
    )

    # Store actual datetime values
    df["order_date"] = parsed_dates

    # Derive month, year and quarter from the parsed dates
    df["order_month"] = df["order_date"].dt.month
    df["order_year"] = df["order_date"].dt.year
    df["order_quarter"] = df["order_date"].dt.quarter

    return df


In [46]:

df_2018_raw = pd.read_csv("../data/raw/amazon_india_2018.csv")

df_2018_test = clean_amazon_dataset(df_2018_raw)

print("Date data type:", df_2018_test["order_date"].dtype)
print("Missing order dates:", df_2018_test["order_date"].isna().sum())


Date data type: str
Missing order dates: 0


In [47]:

print("Month mismatches:",
      (df_2018_test["order_month"]
       != df_2018_test["order_date"].dt.month).sum())

print("Year mismatches:",
      (df_2018_test["order_year"]
       != df_2018_test["order_date"].dt.year).sum())

print("Quarter mismatches:",
      (df_2018_test["order_quarter"]
       != df_2018_test["order_date"].dt.quarter).sum())


AttributeError: Can only use .dt accessor with datetimelike values

In [48]:
df = clean_order_dates(df)

In [50]:

def clean_amazon_dataset(df):
    df = df.copy()

    # 1. Standardize festival flag
    df = clean_festival_flag(df)

    # 2. Clean missing values
    df = clean_missing_values(df)

    # 3. Clean customer ratings
    df = clean_customer_rating(df)

    # 4. Clean delivery charges
    df = clean_delivery_charges(df)

    # 5. Clean original prices
    df = clean_original_price(df)

    # 6. Correct negative original prices
    negative_mask = df["original_price_inr"] < 0

    df.loc[negative_mask, "original_price_inr"] = (
        df.loc[negative_mask, "original_price_inr"].abs()
    )

    # 7. Correct potential scale errors
    scale_error_mask = (
        df["original_price_inr"].notna()
        & df["discounted_price_inr"].notna()
        & (
            (
                df["original_price_inr"] / 100
                - df["discounted_price_inr"]
            ).abs() < 1
        )
    )

    df.loc[scale_error_mask, "original_price_inr"] = (
        df.loc[scale_error_mask, "original_price_inr"] / 100
    )

    # 8. Standardize product categories
    df = standardize_categories(df)

    # 9. Standardize customer cities
    df = standardize_cities(df)

    # 10. Clean and convert order dates
    df = clean_order_dates(df)

    return df


In [51]:

df_2018_raw = pd.read_csv("../data/raw/amazon_india_2018.csv")

df_2018_test = clean_amazon_dataset(df_2018_raw)

print("Raw shape:", df_2018_raw.shape)
print("Cleaned shape:", df_2018_test.shape)
print("Date data type:", df_2018_test["order_date"].dtype)
print("Missing order dates:", df_2018_test["order_date"].isna().sum())


Raw shape: (99495, 34)
Cleaned shape: (99495, 34)
Date data type: datetime64[us]
Missing order dates: 0


In [52]:
print("Date data type:", df_2018_test["order_date"].dtype)

print("Missing order dates:",
      df_2018_test["order_date"].isna().sum())

print("Month mismatches:",
      (df_2018_test["order_month"]
       != df_2018_test["order_date"].dt.month).sum())

print("Year mismatches:",
      (df_2018_test["order_year"]
       != df_2018_test["order_date"].dt.year).sum())

print("Quarter mismatches:",
      (df_2018_test["order_quarter"]
       != df_2018_test["order_date"].dt.quarter).sum())

Date data type: datetime64[us]
Missing order dates: 0
Month mismatches: 0
Year mismatches: 0
Quarter mismatches: 0


In [53]:

cleaning_results = []

for file_path in raw_files:
    # Read one raw dataset
    df_raw = pd.read_csv(file_path)

    # Clean the dataset
    df_clean = clean_amazon_dataset(df_raw)

    # Create the cleaned output filename
    output_path = cleaned_folder / f"{file_path.stem}_clean.csv"

    # Save the cleaned dataset
    df_clean.to_csv(output_path, index=False)

    # Record validation results
    cleaning_results.append({
        "file": file_path.name,
        "rows": len(df_clean),
        "columns": len(df_clean.columns),
        "missing_dates": df_clean["order_date"].isna().sum(),
        "negative_prices": (
            df_clean["original_price_inr"] < 0
        ).sum(),
        "duplicate_transactions": (
            df_clean["transaction_id"].duplicated().sum()
        ),
        "saved_to": str(output_path)
    })

# Display the summary
cleaning_summary = pd.DataFrame(cleaning_results)

print(cleaning_summary.to_string(index=False))


                 file   rows  columns  missing_dates  negative_prices  duplicate_transactions                                    saved_to
amazon_india_2015.csv  33165       34              0                0                       0 ..\data\cleaned\amazon_india_2015_clean.csv
amazon_india_2016.csv  55275       34              0                0                       0 ..\data\cleaned\amazon_india_2016_clean.csv
amazon_india_2017.csv  77385       34              0                0                       0 ..\data\cleaned\amazon_india_2017_clean.csv
amazon_india_2018.csv  99495       34              0                0                       0 ..\data\cleaned\amazon_india_2018_clean.csv
amazon_india_2019.csv 121605       34              0                0                       0 ..\data\cleaned\amazon_india_2019_clean.csv
amazon_india_2020.csv 143715       34              0                0                       0 ..\data\cleaned\amazon_india_2020_clean.csv
amazon_india_2021.csv 138187      

In [54]:

validation_results = []

for file_path in sorted(cleaned_folder.glob("amazon_india_*_clean.csv")):
    df = pd.read_csv(file_path)

    # Convert date column for validation
    df["order_date"] = pd.to_datetime(
        df["order_date"], errors="coerce"
    )

    validation_results.append({
        "file": file_path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values_total": int(df.isna().sum().sum()),
        "missing_dates": int(df["order_date"].isna().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_transaction_ids": int(
            df["transaction_id"].duplicated().sum()
        ),
        "invalid_customer_ratings": int(
            (~df["customer_rating"].dropna().between(0, 5)).sum()
        ),
        "negative_original_prices": int(
            (df["original_price_inr"] < 0).sum()
        )
    })

validation_summary = pd.DataFrame(validation_results)

print(validation_summary.to_string(index=False))


                       file   rows  columns  missing_values_total  missing_dates  duplicate_rows  duplicate_transaction_ids  invalid_customer_ratings  negative_original_prices
amazon_india_2015_clean.csv  33165       34                 10956              0               0                          0                         0                         0
amazon_india_2016_clean.csv  55275       34                 18399              0               0                          0                         0                         0
amazon_india_2017_clean.csv  77385       34                 25871              0               0                          0                         0                         0
amazon_india_2018_clean.csv  99495       34                 33216              0               0                          0                         0                         0
amazon_india_2019_clean.csv 121605       34                 40406              0               0                        

In [55]:

# Inspect missing values in the cleaned 2018 dataset
df_2018_check = pd.read_csv(
    "../data/cleaned/amazon_india_2018_clean.csv"
)

missing_report = df_2018_check.isna().sum()

missing_report = missing_report[missing_report > 0].sort_values(
    ascending=False
)

print("Columns with missing values:")
print(missing_report)

print("\nMissing percentage:")
print(
    (missing_report / len(df_2018_check) * 100).round(2)
)


Columns with missing values:
customer_rating       30234
original_price_inr     2982
dtype: int64

Missing percentage:
customer_rating       30.39
original_price_inr     3.00
dtype: float64


In [56]:

missing_price_rows = df_2018_check[
    df_2018_check["original_price_inr"].isna()
]

print("Rows with missing original prices:",
      len(missing_price_rows))

print(missing_price_rows[
    [
        "product_name",
        "original_price_inr",
        "discounted_price_inr",
        "discount_percent",
        "quantity",
        "subtotal_inr",
        "final_amount_inr"
    ]
].head(15))


Rows with missing original prices: 2982
                         product_name  original_price_inr  \
61      OnePlus OnePlus 6T 128GB Blue                 NaN   
190          Oppo R17 Pro 128GB Black                 NaN   
210    Xiaomi Redmi Note 4 16GB White                 NaN   
222      Samsung Galaxy S6 16GB Black                 NaN   
226     OnePlus OnePlus 5T 16GB Black                 NaN   
325           Oppo R17 Pro 64GB Black                 NaN   
327              Vivo Y95 128GB Black                 NaN   
404          Apple iPhone X 16GB Gold                 NaN   
412         Lenovo Pad 8GB RAM Silver                 NaN   
451   Motorola Moto G4 Plus 16GB Blue                 NaN   
496  Samsung Galaxy Note 9 64GB White                 NaN   
505                 Vivo V7 32GB Blue                 NaN   
523      OnePlus OnePlus 3T 16GB Blue                 NaN   
542        Realme Realme 2 64GB White                 NaN   
569       Xiaomi Slate 8GB RAM Silver        

In [57]:

missing_price_rows = df_2018_check[
    df_2018_check["original_price_inr"].isna()
].copy()

# Check rows where discount is zero
zero_discount = missing_price_rows[
    missing_price_rows["discount_percent"] == 0
]

print("Missing original prices:", len(missing_price_rows))
print("Rows with zero discount:", len(zero_discount))

print(
    zero_discount[
        [
            "product_name",
            "discounted_price_inr",
            "discount_percent",
            "subtotal_inr",
            "quantity"
        ]
    ].head(10)
)


Missing original prices: 2982
Rows with zero discount: 1318
                          product_name  discounted_price_inr  \
210     Xiaomi Redmi Note 4 16GB White              30661.00   
325            Oppo R17 Pro 64GB Black              26128.66   
412          Lenovo Pad 8GB RAM Silver              94787.00   
505                  Vivo V7 32GB Blue              48227.04   
523       OnePlus OnePlus 3T 16GB Blue              97072.39   
542         Realme Realme 2 64GB White              23354.55   
569        Xiaomi Slate 8GB RAM Silver              28560.11   
581      Apple VivoBook 4GB RAM Silver             138864.20   
595      OnePlus OnePlus 3T 32GB Black             101957.82   
751  Samsung Galaxy Tab 4GB RAM Silver              77883.23   

     discount_percent  subtotal_inr  quantity  
210               0.0      30661.00         1  
325               0.0      26128.66         1  
412               0.0     189574.00         2  
505               0.0      48227.04        

In [58]:

df_2018_raw_check = pd.read_csv(
    "../data/raw/amazon_india_2018.csv"
)

product_price_lookup = (
    df_2018_raw_check[
        df_2018_raw_check["original_price_inr"].notna()
    ]
    .groupby("product_name")["original_price_inr"]
    .agg(["count", "nunique", "first"])
)

missing_product_names = missing_price_rows["product_name"].unique()

print(
    product_price_lookup.loc[
        product_price_lookup.index.isin(missing_product_names)
    ].head(15)
)


                              count  nunique        first
product_name                                             
ASUS Gaming 4GB RAM Black       167        3    141846.11
ASUS Gaming 8GB RAM Silver      252        5     61097.13
ASUS Inspiron 4GB RAM Black     256        6    138070.49
ASUS Inspiron 4GB RAM Silver     50        5     233742.6
ASUS Inspiron 8GB RAM Black     230        6  ₹111,455.61
ASUS Inspiron 8GB RAM Silver    323        6     48472.48
ASUS MacBook 4GB RAM Silver      66        5    159477.91
ASUS Pavilion 4GB RAM Silver    200        4     64766.17
ASUS Ultrabook 4GB RAM Black     40        4    164363.15
ASUS Ultrabook 8GB RAM Black     76        4    223657.81
Acer Gaming 4GB RAM Silver       60        4    197567.74
Acer Gaming 8GB RAM Silver      247        5    151060.56
Acer MacBook 4GB RAM Black      305        4     70447.94
Acer MacBook 4GB RAM Silver     234        5     64874.95
Acer Pavilion 4GB RAM Silver    246        5     94678.29


In [59]:
# Check discounted price against subtotal / quantity

price_check = df_2018_check[
    df_2018_check["quantity"] > 0
].copy()

price_check["calculated_unit_price"] = (
    price_check["subtotal_inr"] / price_check["quantity"]
)

price_check["price_difference"] = (
    price_check["calculated_unit_price"]
    - price_check["discounted_price_inr"]
).abs()

print("Matching unit prices:",
      (price_check["price_difference"] < 0.02).sum())

print("Non-matching unit prices:",
      (price_check["price_difference"] >= 0.02).sum())

Matching unit prices: 99495
Non-matching unit prices: 0


In [60]:
amount_check = df_2018_check[
    df_2018_check["delivery_charges"].notna()
].copy()

amount_check["calculated_final_amount"] = (
    amount_check["subtotal_inr"]
    + amount_check["delivery_charges"]
)

amount_check["amount_difference"] = (
    amount_check["calculated_final_amount"]
    - amount_check["final_amount_inr"]
).abs()

print("Matching final amounts:",
      (amount_check["amount_difference"] < 0.02).sum())

print("Non-matching final amounts:",
      (amount_check["amount_difference"] >= 0.02).sum())

Matching final amounts: 99495
Non-matching final amounts: 0


In [61]:
discount_check = df_2018_check[
    df_2018_check["original_price_inr"].notna()
].copy()

discount_check["calculated_discounted_price"] = (
    discount_check["original_price_inr"]
    * (1 - discount_check["discount_percent"] / 100)
)

discount_check["discount_difference"] = (
    discount_check["calculated_discounted_price"]
    - discount_check["discounted_price_inr"]
).abs()

print(
    "Matching discounted prices:",
    (discount_check["discount_difference"] < 1).sum()
)

print(
    "Non-matching discounted prices:",
    (discount_check["discount_difference"] >= 1).sum()
)

Matching discounted prices: 66403
Non-matching discounted prices: 30110


In [62]:
mismatched_discount = discount_check[
    discount_check["discount_difference"] >= 1
].copy()

print(
    mismatched_discount[
        [
            "product_name",
            "original_price_inr",
            "discount_percent",
            "discounted_price_inr",
            "calculated_discounted_price",
            "discount_difference"
        ]
    ].head(15).to_string(index=False)
)

                     product_name  original_price_inr  discount_percent  discounted_price_inr  calculated_discounted_price  discount_difference
         Apple iPhone X 64GB Blue           201630.77             47.20             106459.69                106461.046560             1.356560
                     Fitbit Watch            59337.94             46.61              31682.93                 31680.526166             2.403834
 Samsung Galaxy Note 9 128GB Blue           142248.89             28.42             101828.63                101821.755462             6.874538
                Vivo V7 16GB Blue            49034.97             67.65              15865.00                 15862.812795             2.187205
     ASUS Inspiron 8GB RAM Silver            48472.48             20.58              38499.10                 38496.843616             2.256384
    Samsung Galaxy S9+ 256GB Blue            87376.17             25.54              65055.93                 65060.296182             4

In [63]:
print("Discount percentage range:")
print(discount_check["discount_percent"].describe())

print("\nMismatch difference statistics:")
print(mismatched_discount["discount_difference"].describe())

Discount percentage range:
count    96513.000000
mean        17.658450
std         20.670472
min          0.000000
25%          0.000000
50%         10.820000
75%         28.750000
max         70.000000
Name: discount_percent, dtype: float64

Mismatch difference statistics:
count    3.011000e+04
mean     2.403283e+04
std      4.156129e+05
min      1.000008e+00
25%      1.420939e+00
50%      2.144679e+00
75%      3.741415e+00
max      2.018916e+07
Name: discount_difference, dtype: float64


In [64]:
large_mismatches = mismatched_discount.sort_values(
    by="discount_difference",
    ascending=False
)

print(
    large_mismatches[
        [
            "product_name",
            "original_price_inr",
            "discount_percent",
            "discounted_price_inr",
            "calculated_discounted_price",
            "discount_difference"
        ]
    ].head(20).to_string(index=False)
)

                    product_name  original_price_inr  discount_percent  discounted_price_inr  calculated_discounted_price  discount_difference
  Apple iPhone XS Max 64GB White          24237111.0             15.86             203941.35                 2.039311e+07         2.018916e+07
                     LG Smart TV          20101392.0              5.57             189821.42                 1.898174e+07         1.879192e+07
     Apple iPhone XR 128GB Black          23465890.0             19.65             188539.48                 1.885484e+07         1.866630e+07
  Apple iPhone 6 Plus 64GB White          19953359.0             18.24             163138.84                 1.631387e+07         1.615073e+07
     ASUS Pavilion 4GB RAM Black          20493642.0             21.68             160506.59                 1.605062e+07         1.589011e+07
Samsung Galaxy Note 9 64GB Black          18779798.0             26.07             138834.96                 1.388390e+07         1.374507e+07

In [65]:
print(
    df_2018_check["original_price_inr"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

count    9.651300e+04
mean     8.107996e+04
std      3.241954e+05
min      2.202130e+03
25%      3.101855e+04
50%      4.873308e+04
75%      1.045926e+05
90%      1.485089e+05
95%      1.807990e+05
99%      2.413400e+05
max      2.423711e+07
Name: original_price_inr, dtype: float64


In [66]:
# Create a working copy
df_2018_final = df_2018_check.copy()

# Calculate expected discounted price after dividing original price by 100
corrected_discounted_price = (
    (df_2018_final["original_price_inr"] / 100)
    * (1 - df_2018_final["discount_percent"] / 100)
)

# Identify prices where the corrected calculation matches
# the actual discounted price within Rs. 1
scale_error_mask = (
    df_2018_final["original_price_inr"].notna()
    & df_2018_final["discounted_price_inr"].notna()
    & df_2018_final["discount_percent"].between(0, 100)
    & (
        corrected_discounted_price
        - df_2018_final["discounted_price_inr"]
    ).abs().lt(1)
)

# Correct only verified records
df_2018_final.loc[
    scale_error_mask, "original_price_inr"
] = (
    df_2018_final.loc[
        scale_error_mask, "original_price_inr"
    ] / 100
)

# Validate discount calculation again
df_2018_final["calculated_discounted_price"] = (
    df_2018_final["original_price_inr"]
    * (1 - df_2018_final["discount_percent"] / 100)
)

df_2018_final["discount_difference"] = (
    df_2018_final["calculated_discounted_price"]
    - df_2018_final["discounted_price_inr"]
).abs()

print("Corrected scale-error records:", scale_error_mask.sum())

print(
    "Matching discount calculations:",
    (df_2018_final["discount_difference"] < 1).sum()
)

print(
    "Non-matching discount calculations:",
    (df_2018_final["discount_difference"] >= 1).sum()
)

Corrected scale-error records: 58
Matching discount calculations: 66461
Non-matching discount calculations: 30052


In [67]:
df_2018_final.drop(
    columns=[
        "calculated_discounted_price",
        "discount_difference"
    ]
).to_csv(
    "../data/cleaned/amazon_india_2018_clean.csv",
    index=False
)

In [68]:
# Correct verified 100x original-price errors
corrected_discounted_price = (
    (df["original_price_inr"] / 100)
    * (1 - df["discount_percent"] / 100)
)

scale_error_mask = (
    df["original_price_inr"].notna()
    & df["discounted_price_inr"].notna()
    & df["discount_percent"].between(0, 100)
    & (
        corrected_discounted_price
        - df["discounted_price_inr"]
    ).abs().lt(1)
)

df.loc[scale_error_mask, "original_price_inr"] /= 100

In [69]:
cleaned_files = sorted(
    cleaned_folder.glob("amazon_india_*_clean.csv")
)

all_years = []

for file_path in cleaned_files:
    temp = pd.read_csv(
        file_path,
        parse_dates=["order_date"]
    )
    all_years.append(temp)

amazon_all = pd.concat(
    all_years,
    ignore_index=True
)

print("Combined dataset shape:", amazon_all.shape)

print("\nRecords by year:")
print(amazon_all["order_year"].value_counts().sort_index())

print("\nTotal missing values:")
print(amazon_all.isnull().sum().sum())

Combined dataset shape: (1127609, 34)

Records by year:
order_year
2015     33165
2016     55275
2017     77385
2018     99495
2019    121605
2020    143715
2021    138187
2022    132660
2023    127132
2024    121605
2025     77385
Name: count, dtype: int64

Total missing values:
375327


In [70]:
# Save the combined master dataset
master_file = cleaned_folder / "amazon_master_2015_2025.csv"

amazon_all.to_csv(master_file, index=False)

print("MASTER DATASET SAVED SUCCESSFULLY")
print("File:", master_file)
print("Total Records:", len(amazon_all))
print("Total Columns:", len(amazon_all.columns))
print("Years Covered:", amazon_all["order_year"].nunique())
print("Total Missing Values:", amazon_all.isnull().sum().sum())

MASTER DATASET SAVED SUCCESSFULLY
File: ..\data\cleaned\amazon_master_2015_2025.csv
Total Records: 1127609
Total Columns: 34
Years Covered: 11
Total Missing Values: 375327
